# ⏩ Microsoft Foundry（Python）を使った連続エージェントワークフロー

## 📋 高度な連続処理チュートリアル

このノートブックは Microsoft Agent Framework を使った<strong>連続ワークフローパターン</strong>を示します。エージェントが特定の順序で実行され、ステージ間でデータやコンテキストを渡す高度な多段階処理パイプラインの構築方法を学びます。

> **移行メモ:** 以前は GitHub Models を参照していましたが、GitHub Models は非推奨（2026年7月に終了予定）となったため、現在は `FoundryChatClient` を通じて Azure OpenAI の **Responses API** をターゲットにした **Microsoft Foundry** を使用しています。

## 🎯 学習目標

### 🔄 <strong>連続処理パターン</strong>
- <strong>線形ワークフローデザイン</strong>: ステップバイステップの処理パイプラインを作成
- <strong>データフロー管理</strong>: 連続エージェント間で情報を渡す
- <strong>ステージゲート処理</strong>: チェックポイントと検証段階の実装
- <strong>進捗追跡</strong>: ワークフローの実行と中間結果を監視

### 🏗️ <strong>エンタープライズパイプラインアーキテクチャ</strong>
- <strong>ビジネスプロセスモデリング</strong>: 実際のビジネスプロセスをエージェントワークフローにマップ
- <strong>品質保証</strong>: 多段階の検証とレビュー処理
- <strong>ドキュメント処理</strong>: 連続したドキュメント分析と変換
- <strong>コンテンツ制作</strong>: レビューと承認段階を含む編集ワークフロー

### 📊 <strong>高度なワークフロー機能</strong>
- <strong>コンテキスト保持</strong>: ワークフローステージ間の状態維持
- <strong>エラー伝播</strong>: 連続処理の失敗をハンドリング
- <strong>パフォーマンス最適化</strong>: 効率的な連続実行パターン
- <strong>監査証跡</strong>: 連続操作の完全な追跡

## ⚙️ 前提条件とセットアップ

### 📦 <strong>依存関係</strong>
```bash
pip install agent-framework -U
```

### 🔑 <strong>設定</strong>

`AzureCliCredential` が認証できるように Azure CLI (`az login`) でサインインし、Microsoft Foundry のプロジェクト詳細を設定します。

```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

## 🏢 <strong>エンタープライズ連続ワークフローのユースケース</strong>

### 📝 <strong>ドキュメント処理パイプライン</strong>
```
Raw Document → Content Extraction → Analysis → Validation → Final Output
```

### 🔍 <strong>品質保証ワークフロー</strong>
```
Initial Review → Technical Validation → Compliance Check → Final Approval
```

### 📰 <strong>コンテンツ制作パイプライン</strong>
```
Research → Writing → Editing → Review → Publishing
```

### 💼 <strong>ビジネスプロセス自動化</strong>
```
Data Collection → Processing → Analysis → Report Generation → Distribution
```

## 🎨 <strong>連続ワークフローデザインの原則</strong>

- **🔗 線形進行**: 各ステージは前のステージの出力に依存
- **📋 状態管理**: 全ステージ間でコンテキストとデータを保存
- **🛡️ エラーハンドリング**: 任意のステージでの優雅な失敗管理
- **📊 進捗監視**: 各ステージの完了とパフォーマンスを追跡
- **🔄 ステージの再利用可能性**: 再利用可能なワークフローコンポーネントを設計

洗練された連続処理ワークフローを構築しましょう！🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
from agent_framework import (
    WorkflowBuilder,
    WorkflowEvent,
    WorkflowViz,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential


In [ ]:

import os
import base64
from dotenv import load_dotenv

In [ ]:
load_dotenv()

In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
SalesAgentName = "Sales-Agent"
SalesAgentInstructions = "You are my furniture sales consultant, you can find different furniture elements from the pictures and give me a purchase suggestion"

In [ ]:
PriceAgentName = "Price-Agent"
PriceAgentInstructions = """You are a furniture pricing specialist and budget consultant. Your responsibilities include:
        1. Analyze furniture items and provide realistic price ranges based on quality, brand, and market standards
        2. Break down pricing by individual furniture pieces
        3. Provide budget-friendly alternatives and premium options
        4. Consider different price tiers (budget, mid-range, premium)
        5. Include estimated total costs for room setups
        6. Suggest where to find the best deals and shopping recommendations
        7. Factor in additional costs like delivery, assembly, and accessories
        8. Provide seasonal pricing insights and best times to buy
        Always format your response with clear price breakdowns and explanations for the pricing rationale."""

In [ ]:
QuoteAgentName = "Quote-Agent"
QuoteAgentInstructions = """You are a assistant that create a quote for furniture purchase.
        1. Create a well-structured quote document that includes:
        2. A title page with the document title, date, and client name
        3. An introduction summarizing the purpose of the document
        4. A summary section with total estimated costs and recommendations
        5. Use clear headings, bullet points, and tables for easy readability
        6. All quotes are presented in markdown form"""

In [ ]:
sales_agent = provider.as_agent(
    name=SalesAgentName,
    instructions=SalesAgentInstructions,
)

price_agent = provider.as_agent(
    name=PriceAgentName,
    instructions=PriceAgentInstructions,
)

quote_agent = provider.as_agent(
    name=QuoteAgentName,
    instructions=QuoteAgentInstructions,
)


In [ ]:
workflow = (
    WorkflowBuilder(start_executor=sales_agent)
    .add_edge(sales_agent, price_agent)
    .add_edge(price_agent, quote_agent)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra (`pip install graphviz`) plus the
# graphviz system binary; if it's not available, fall back to the text strings above.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
class DatabaseEvent(WorkflowEvent): ...

In [ ]:
# Display the exported workflow SVG inline in the notebook

from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
image_path = "../imgs/home.png"
with open(image_path, "rb") as image_file:
    image_b64 = base64.b64encode(image_file.read()).decode()
image_uri = f"data:image/png;base64,{image_b64}"


In [ ]:
# Note: the original notebook used a multimodal message with an image of a
# living room. To keep the lesson focused on sequential workflow mechanics, this
# migration passes a textual description of the same scene as the workflow input.
# Agents accept a plain string, matching the basic and concurrent samples.
message = (
    "I am furnishing a modern living room and want pieces that fit a warm, "
    "inviting style: a comfortable three-seat sofa, two accent armchairs, a "
    "wooden coffee table, a TV stand, a floor lamp, and a soft area rug. "
    "Please find appropriate furniture and give the corresponding price for "
    "each piece, then produce a final purchase quote."
)

In [ ]:
# Workflow.run_stream is no longer part of the public API; the current Workflow
# returns a results object whose `get_outputs()` produces the AgentResponse from
# each output executor. The final stage (quote_agent) is the only output here.
events = await workflow.run(message)
outputs = events.get_outputs()
result = outputs[0].text if outputs else ""

In [ ]:
result.replace("None", "")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**免責事項**：
本書類は AI 翻訳サービス [Co-op Translator](https://github.com/Azure/co-op-translator) を使用して翻訳されています。正確性を期していますが、自動翻訳には誤りや不正確な部分が含まれる可能性があることをご承知おきください。原文の原語版が正式な情報源とみなされるべきです。重要な情報については、専門の人間による翻訳を推奨します。本翻訳の利用により生じたいかなる誤解や解釈違いについても、当方は責任を負いかねます。
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
